In [0]:
# --- CMD 1: PRODUCTION BRONZE LAYER INGESTION ---
print("Streaming raw data rows from Catalog Explorer...")

# Point Spark directly to your live visual table
bronze_df = spark.read.table("workspace.default.real_ai_audit")

print(f"=== BRONZE LAYER: LOADED {bronze_df.count()} RAW COMPUTER VISION LOGS ===")
display(bronze_df)

Streaming raw data rows from Catalog Explorer...
=== BRONZE LAYER: LOADED 48 RAW COMPUTER VISION LOGS ===


timestamp,gate_id,track_id,vehicle_id,confidence
2026-05-20T18:53:27.000Z,GATE_NORTH_01,1,8525,0.6527771949768066
2026-05-20T18:53:29.000Z,GATE_NORTH_01,2,KW527,0.9953413009643555
2026-05-20T18:53:56.000Z,GATE_NORTH_01,21,29,0.5110655426979065
2026-05-20T18:53:57.000Z,GATE_NORTH_01,46,K,0.5714686512947083
2026-05-20T18:54:02.000Z,GATE_NORTH_01,47,88112,0.6532393097877502
2026-05-20T18:54:06.000Z,GATE_NORTH_01,19,OSFS,0.6003933548927307
2026-05-20T18:54:11.000Z,GATE_NORTH_01,52,82,0.5893496870994568
2026-05-20T18:54:47.000Z,GATE_NORTH_01,87,TAXI,0.8194205164909363
2026-05-20T18:54:50.000Z,GATE_NORTH_01,80,3&72,0.6222250461578369
2026-05-20T18:54:53.000Z,GATE_NORTH_01,55,8112,0.856921911239624


In [0]:
# --- CMD 2: SILVER LAYER DATA PURIFICATION ---
from pyspark.sql.functions import col, upper, length

print("Running enterprise cleaning constraints...")

# Filter out low-confidence reads and character fragments
silver_clean_df = bronze_df \
    .filter(col("confidence") >= 0.65) \
    .filter(length(col("vehicle_id")) >= 3) \
    .withColumn("vehicle_id", upper(col("vehicle_id")))

print(f"=== SILVER LAYER: SANITIZED FLEET DATA ({silver_clean_df.count()} ROWS REMAIN) ===")
display(silver_clean_df)

Running enterprise cleaning constraints...
=== SILVER LAYER: SANITIZED FLEET DATA (26 ROWS REMAIN) ===


timestamp,gate_id,track_id,vehicle_id,confidence
2026-05-20T18:53:27.000Z,GATE_NORTH_01,1,8525,0.6527771949768066
2026-05-20T18:53:29.000Z,GATE_NORTH_01,2,KW527,0.9953413009643555
2026-05-20T18:54:02.000Z,GATE_NORTH_01,47,88112,0.6532393097877502
2026-05-20T18:54:47.000Z,GATE_NORTH_01,87,TAXI,0.8194205164909363
2026-05-20T18:54:53.000Z,GATE_NORTH_01,55,8112,0.856921911239624
2026-05-20T18:54:54.000Z,GATE_NORTH_01,85,20112,0.7275603413581848
2026-05-20T18:55:12.000Z,GATE_NORTH_01,131,VIDE,0.8730243444442749
2026-05-20T18:55:23.000Z,GATE_NORTH_01,154,112,0.9944078922271729
2026-05-20T18:55:46.000Z,GATE_NORTH_01,219,8112,0.838463306427002
2026-05-20T18:55:54.000Z,GATE_NORTH_01,238,THE,0.6732714176177979


In [0]:
# --- CMD 3: GOLD LAYER METRICS & RECONCILIATION ---
from pyspark.sql.functions import col  # Cleanly import col for the select statement

print("Reconciling gate logs with scheduling database...")

# Mocking your company's planned arrivals for this timeframe
scheduled_fleet_data = [
    {"scheduled_id": "KW527", "cargo_manifest": "General Freight"},
    {"scheduled_id": "TAXI", "cargo_manifest": "Priority Courier"},
    {"scheduled_id": "TRK-9999", "cargo_manifest": "High-Value Supply"} # This truck skipped the gate!
]

schedule_df = spark.createDataFrame(scheduled_fleet_data)

# Perform the audit link
reconciliation_gold_df = schedule_df.join(
    silver_clean_df, 
    schedule_df.scheduled_id == silver_clean_df.vehicle_id, 
    "left"
)

print("=== GOLD LAYER: LOGISTICS AUDIT MANIFEST ===")
display(reconciliation_gold_df.select(
    col("scheduled_id").alias("Expected_Vehicle"),
    col("cargo_manifest").alias("Cargo_Manifest"),
    col("vehicle_id").alias("AI_Visual_Confirmation"),
    col("confidence").alias("Verification_Confidence")
))

# Compute KPIs using local python counting primitives
scheduled_count = schedule_df.count()
visual_count = silver_clean_df.filter(col("vehicle_id").isin(["KW527", "TAXI"])).count()

# FIXED: Explicitly use Python's built-in absolute value function to avoid conflict
import builtins
ids_score = builtins.abs(visual_count - scheduled_count) / float(scheduled_count)

print("====================================================")
print(f"       AUDITOR EXECUTIVE METRICS COMMAND CENTER    ")
print("====================================================")
print(f" Total Fleet Vehicles Scheduled   : {scheduled_count}")
print(f" Total Fleet Vehicles Verified    : {visual_count}")
print(f" INVENTORY DISCREPANCY SCORE (IDS) : {ids_score * 100:.1f}%")
print("----------------------------------------------------")
if ids_score > 0:
    print(" ⚠️  ALERT: Gate mismatch detected! Expected shipments are missing.")
else:
    print(" ✅ SUCCESS: Gate logistics fully synchronized with cloud manifests.")
print("====================================================")

Reconciling gate logs with scheduling database...
=== GOLD LAYER: LOGISTICS AUDIT MANIFEST ===


Expected_Vehicle,Cargo_Manifest,AI_Visual_Confirmation,Verification_Confidence
KW527,General Freight,KW527,0.9953413009643555
TAXI,Priority Courier,TAXI,0.8194205164909363
TRK-9999,High-Value Supply,null,null


       AUDITOR EXECUTIVE METRICS COMMAND CENTER    
 Total Fleet Vehicles Scheduled   : 3
 Total Fleet Vehicles Verified    : 2
 INVENTORY DISCREPANCY SCORE (IDS) : 33.3%
----------------------------------------------------
 ⚠️  ALERT: Gate mismatch detected! Expected shipments are missing.
